# FocalNet-Tiny SRF Training and Evaluation

Notebook-first workflow for the CSC3109 Group 12 aerial-image classification project.

This notebook keeps the FocalNet training and evaluation control flow visible while importing shared project helpers from `src`. The official held-out validation folder is reserved for the final, reportable evaluation only.

## Run protocol

- Use `data/raw/train` for training and the internal tune split.
- Use `data/raw/val` only in the final section marked **RUN ONCE**.
- Run the notebook top-to-bottom for the final pretrained FocalNet-Tiny SRF training run, then run the final held-out evaluation section once.
- Final/reportable results require ImageNet pretrained FocalNet weights; `pretrained=False` is diagnostic only.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import hashlib
import json
import math
import random
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import timm
from torchvision import datasets

from src.config import CLASS_NAMES, MODEL_DIR, REPORTS_DIR, NUM_CLASSES
from src.data import build_eval_transform, build_internal_split_dataloaders
from src.evaluation import (
    classification_metrics,
    save_confusion_matrix_plot,
    write_epoch_history_csv,
    write_metrics_json,
)
from src.models import (
    FOCALNET_TINY_SRF,
    build_timm_classifier,
    get_timm_preprocess_settings,
    resolve_timm_model_name,
    trainable_parameters,
)

## Environment check

**Held-out validation integrity note:** the next dataset sanity cells may read `data/raw/val` only for read-only integrity checks such as class counts and duplicate-image hashes. Held-out validation images are not used for training, tuning, or model selection. Final model evaluation is performed only in the final **RUN ONCE** section.

In [ ]:
print(f'torch: {torch.__version__}')
print(f'timm: {timm.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Dataset configuration and sanity checks

In [ ]:
TRAIN_DIR = PROJECT_ROOT / 'data' / 'raw' / 'train'
HELDOUT_VAL_DIR = PROJECT_ROOT / 'data' / 'raw' / 'val'

assert TRAIN_DIR.exists(), f'Missing training directory: {TRAIN_DIR}'
assert HELDOUT_VAL_DIR.exists(), f'Missing held-out validation directory: {HELDOUT_VAL_DIR}'
print(f'TRAIN_DIR = {TRAIN_DIR}')
print(f'HELDOUT_VAL_DIR = {HELDOUT_VAL_DIR}')

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def image_paths_by_class(root: Path) -> dict[str, list[Path]]:
    paths = {}
    for class_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        paths[class_dir.name] = sorted(
            path for path in class_dir.rglob('*') if path.suffix.lower() in IMAGE_EXTENSIONS
        )
    return paths

train_paths = image_paths_by_class(TRAIN_DIR)
heldout_paths = image_paths_by_class(HELDOUT_VAL_DIR)
counts = pd.DataFrame({
    'class_name': sorted(set(train_paths) | set(heldout_paths)),
}).assign(
    train_count=lambda df: df['class_name'].map(lambda name: len(train_paths.get(name, []))),
    heldout_val_count=lambda df: df['class_name'].map(lambda name: len(heldout_paths.get(name, []))),
)
display(counts)
assert set(counts['class_name']) == set(CLASS_NAMES), f'Unexpected classes: {counts}'

In [ ]:
rng = random.Random(42)
sample_paths = []
for class_name in CLASS_NAMES:
    candidates = train_paths.get(class_name, [])
    sample_paths.extend(rng.sample(candidates, k=min(3, len(candidates))))

cols = 4
rows = max(1, math.ceil(len(sample_paths) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
axes = np.atleast_1d(axes).ravel()
for ax, path in zip(axes, sample_paths):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(path.parent.name)
    ax.axis('off')
for ax in axes[len(sample_paths):]:
    ax.axis('off')
fig.suptitle('Training samples from data/raw/train')
fig.tight_layout()

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

train_hashes = {sha256_file(path): path for paths in train_paths.values() for path in paths}
heldout_hashes = {sha256_file(path): path for paths in heldout_paths.values() for path in paths}
duplicate_hashes = sorted(set(train_hashes) & set(heldout_hashes))
print(f'Cross-split duplicate hashes: {len(duplicate_hashes)}')
if duplicate_hashes:
    display(pd.DataFrame([
        {'hash': h, 'train_path': str(train_hashes[h]), 'heldout_val_path': str(heldout_hashes[h])}
        for h in duplicate_hashes[:20]
    ]))
assert not duplicate_hashes, 'Found duplicate image bytes across train and held-out validation splits.'

## FocalNet run configuration

In [ ]:
MODEL_NAME = 'focalnet-tiny-srf'
SEED = 42
IMAGE_SIZE = 224
EPOCHS = 20
BATCH_SIZE = 16
NUM_WORKERS = 0
LR = 3e-5
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
TUNE_RATIO = 0.2
PATIENCE = 5
MIN_DELTA = 1e-4

RUN_LABEL = 'final'
RUN_DIR = MODEL_DIR / 'focalnet_tiny_srf_notebook' / RUN_LABEL
OUTPUT_DIR = RUN_DIR
REPORT_DIR = REPORTS_DIR / 'focalnet_tiny_srf_notebook_eval'
MAX_TRAIN_BATCHES = None
MAX_TUNE_BATCHES = None
EPOCHS_TO_RUN = EPOCHS
PRETRAINED = True

assert MODEL_NAME == FOCALNET_TINY_SRF.alias
assert PRETRAINED is True, 'Final/reportable FocalNet runs must use ImageNet pretrained weights.'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run label: {RUN_LABEL}')
print(f'Pretrained: {PRETRAINED}, epochs_to_run={EPOCHS_TO_RUN}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Report dir: {REPORT_DIR}')

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Pretrained availability and model construction

In [ ]:
resolved_model_name = resolve_timm_model_name(MODEL_NAME)
available = resolved_model_name in timm.list_models(resolved_model_name)
preprocess = get_timm_preprocess_settings(MODEL_NAME)
print(f'Alias: {MODEL_NAME}')
print(f'timm model: {resolved_model_name}')
print(f'Available in timm registry: {available}')
print(f'Preprocess: {preprocess}')
assert available, f'{resolved_model_name} is not available in this timm install.'

model = build_timm_classifier(
    num_classes=NUM_CLASSES,
    model_name=MODEL_NAME,
    pretrained=PRETRAINED,
    image_size=IMAGE_SIZE,
).to(device)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in trainable_parameters(model))
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## Internal train/tune loaders from `data/raw/train` only

In [ ]:
mean = tuple(float(value) for value in preprocess['mean'])
std = tuple(float(value) for value in preprocess['std'])
interpolation = str(preprocess['interpolation'])

train_loader, tune_loader, class_to_idx = build_internal_split_dataloaders(
    train_dir=TRAIN_DIR,
    tune_ratio=TUNE_RATIO,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == 'cuda',
    seed=SEED,
    mean=mean,
    std=std,
    interpolation=interpolation,
)
class_names = [name for name, _ in sorted(class_to_idx.items(), key=lambda item: item[1])]
print(f'Classes: {class_names}')
print(f'Train images: {len(train_loader.dataset)}')
print(f'Tune images: {len(tune_loader.dataset)}')
assert class_names == list(CLASS_NAMES)

## Notebook-visible training and tune evaluation helpers

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch: int, max_batches: int | None = None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    progress = tqdm(loader, desc=f'Epoch {epoch} train', leave=False)
    for batch_index, (images, labels) in enumerate(progress, start=1):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += batch_size
        progress.set_postfix(loss=running_loss / max(total, 1), acc=correct / max(total, 1))
        if max_batches is not None and batch_index >= max_batches:
            break
    return running_loss / max(total, 1), correct / max(total, 1)

@torch.no_grad()
def evaluate_loop(model, loader, criterion, device, phase: str = 'tune', max_batches: int | None = None):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    y_true, y_pred = [], []
    progress = tqdm(loader, desc=phase, leave=False)
    for batch_index, (images, labels) in enumerate(progress, start=1):
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (predictions == labels).sum().item()
        total += batch_size
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(predictions.cpu().tolist())
        progress.set_postfix(loss=running_loss / max(total, 1), acc=correct / max(total, 1))
        if max_batches is not None and batch_index >= max_batches:
            break
    return running_loss / max(total, 1), correct / max(total, 1), y_true, y_pred

## Train with macro-F1 checkpoint selection and early stopping

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(list(trainable_parameters(model)), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_TO_RUN)

run_config = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_name': MODEL_NAME,
    'resolved_model_name': resolved_model_name,
    'pretrained': PRETRAINED,
    'seed': SEED,
    'image_size': IMAGE_SIZE,
    'epochs': EPOCHS,
    'epochs_to_run': EPOCHS_TO_RUN,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'tune_ratio': TUNE_RATIO,
    'patience': PATIENCE,
    'min_delta': MIN_DELTA,
    'train_dir': str(TRAIN_DIR),
    'heldout_val_dir': str(HELDOUT_VAL_DIR),
    'preprocess': preprocess,
    'class_to_idx': class_to_idx,
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')

best_macro_f1 = -1.0
best_metrics = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS_TO_RUN + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, MAX_TRAIN_BATCHES)
    tune_loss, tune_acc, y_true, y_pred = evaluate_loop(model, tune_loader, criterion, device, 'tune', MAX_TUNE_BATCHES)
    metrics = classification_metrics(y_true, y_pred, class_names)
    row = {
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'tune_loss': tune_loss,
        'tune_accuracy': tune_acc,
        'tune_macro_precision': metrics['macro_precision'],
        'tune_macro_recall': metrics['macro_recall'],
        'tune_macro_f1': metrics['macro_f1'],
    }
    history.append(row)
    print(row)

    improved = metrics['macro_f1'] > best_macro_f1 + MIN_DELTA
    if improved:
        best_macro_f1 = metrics['macro_f1']
        best_metrics = metrics
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'class_to_idx': class_to_idx,
            'idx_to_class': {index: name for name, index in class_to_idx.items()},
            'metrics': metrics,
            'model_name': MODEL_NAME,
            'resolved_model_name': resolved_model_name,
            'image_size': IMAGE_SIZE,
            'preprocess': preprocess,
            'run_config': run_config,
        }, OUTPUT_DIR / 'best_model.pt')
        write_metrics_json(metrics, OUTPUT_DIR / 'best_tune_metrics.json')
        save_confusion_matrix_plot(metrics['confusion_matrix'], class_names, OUTPUT_DIR / 'best_tune_confusion_matrix.png', title='FocalNet-Tiny SRF best tune confusion matrix')
    else:
        epochs_without_improvement += 1

    scheduler.step()
    if PATIENCE > 0 and epochs_without_improvement >= PATIENCE:
        print(f'Early stopping after {epoch} epochs without macro-F1 improvement.')
        break

write_epoch_history_csv(history, OUTPUT_DIR / 'history.csv')
print(f'Best tune macro-F1: {best_macro_f1:.4f}')

## Tune history and confusion matrix

In [ ]:
history_path = OUTPUT_DIR / 'history.csv'
if history_path.exists():
    history_df = pd.read_csv(history_path)
    display(history_df)
    ax = history_df.plot(x='epoch', y=['train_loss', 'tune_loss'], marker='o', figsize=(8, 4))
    ax.set_title('FocalNet training/tune loss')
    plt.show()
    ax = history_df.plot(x='epoch', y=['train_accuracy', 'tune_accuracy', 'tune_macro_f1'], marker='o', figsize=(8, 4))
    ax.set_title('FocalNet tune history')
    plt.show()

cm_path = OUTPUT_DIR / 'best_tune_confusion_matrix.png'
if cm_path.exists():
    display(Image.open(cm_path))

## Final held-out evaluation — RUN ONCE

This section is the only place where `data/raw/val` is evaluated. Run it once after the final pretrained top-to-bottom training run has produced `best_model.pt`.

In [ ]:
assert RUN_LABEL == 'final', 'Final held-out evaluation requires the final run configuration.'
assert PRETRAINED is True, 'Final/reportable held-out evaluation requires ImageNet pretrained FocalNet.'
checkpoint_path = OUTPUT_DIR / 'best_model.pt'
assert checkpoint_path.exists(), f'Missing checkpoint: {checkpoint_path}'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

heldout_dataset = datasets.ImageFolder(
    HELDOUT_VAL_DIR,
    transform=build_eval_transform(IMAGE_SIZE, mean=mean, std=std, interpolation=interpolation),
)
assert heldout_dataset.class_to_idx == checkpoint['class_to_idx'], (heldout_dataset.class_to_idx, checkpoint['class_to_idx'])
heldout_loader = DataLoader(heldout_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda')

final_model = build_timm_classifier(
    num_classes=len(class_names),
    model_name=checkpoint['model_name'],
    pretrained=False,
    image_size=int(checkpoint['image_size']),
).to(device)
final_model.load_state_dict(checkpoint['model_state_dict'])
final_criterion = nn.CrossEntropyLoss(label_smoothing=0.0)
heldout_loss, heldout_acc, heldout_true, heldout_pred = evaluate_loop(final_model, heldout_loader, final_criterion, device, 'held-out validation', None)
heldout_metrics = classification_metrics(heldout_true, heldout_pred, class_names)
heldout_metrics['loss'] = float(heldout_loss)
heldout_metrics['source_data_dir'] = 'data/raw/val'
heldout_metrics['checkpoint_path'] = 'model/focalnet_tiny_srf_notebook/final/best_model.pt'

write_metrics_json(heldout_metrics, REPORT_DIR / 'metrics.json')
save_confusion_matrix_plot(heldout_metrics['confusion_matrix'], class_names, REPORT_DIR / 'confusion_matrix.png', title='FocalNet-Tiny SRF held-out confusion matrix')
display(pd.DataFrame([heldout_metrics])[['accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'loss']])
display(Image.open(REPORT_DIR / 'confusion_matrix.png'))
print('Wrote final held-out evaluation to reports/focalnet_tiny_srf_notebook_eval')

## Comparison table from existing metric JSON files

In [ ]:
metric_rows = []
for metrics_path in sorted(REPORTS_DIR.glob('*/metrics.json')):
    artifact_name = metrics_path.parent.name
    try:
        payload = json.loads(metrics_path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        continue
    source_data_dir = payload.get('source_data_dir')
    if source_data_dir is not None and Path(source_data_dir) != Path('data/raw/val') and Path(source_data_dir) != HELDOUT_VAL_DIR:
        continue
    metric_rows.append({
        'artifact': artifact_name,
        'accuracy': payload.get('accuracy'),
        'macro_precision': payload.get('macro_precision'),
        'macro_recall': payload.get('macro_recall'),
        'macro_f1': payload.get('macro_f1'),
        'source_path': str(metrics_path.relative_to(PROJECT_ROOT)),
    })
comparison_df = pd.DataFrame(metric_rows).sort_values('macro_f1', ascending=False, na_position='last') if metric_rows else pd.DataFrame()
display(comparison_df)

## Experiment log row generation

In [ ]:
final_metrics_path = REPORTS_DIR / 'focalnet_tiny_srf_notebook_eval' / 'metrics.json'
if final_metrics_path.exists():
    metrics = json.loads(final_metrics_path.read_text(encoding='utf-8'))
    log_row = (
        f"| FocalNet-Tiny SRF notebook | {MODEL_NAME} | data/raw/train internal tune + data/raw/val held-out | "
        f"{metrics['accuracy']:.4f} | {metrics['macro_precision']:.4f} | {metrics['macro_recall']:.4f} | {metrics['macro_f1']:.4f} | "
        f"{final_metrics_path.relative_to(PROJECT_ROOT)} |"
    )
    print('Suggested experiments/results-log.md row:')
    print(log_row)
else:
    print(f"No final held-out metrics found yet: {final_metrics_path.relative_to(PROJECT_ROOT)}")